In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [10]:
MODEL = 'llama3.2'
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [5]:
links = fetch_website_links("https://anthropic.com")

links

['#main',
 '#footer',
 'https://www.anthropic.com/',
 'https://www.anthropic.com/research',
 'https://www.anthropic.com/economic-futures',
 'https://www.anthropic.com/constitution',
 'https://www.anthropic.com/transparency',
 'https://www.anthropic.com/responsible-scaling-policy',
 'http://trust.anthropic.com/',
 'https://www.anthropic.com/learn',
 'https://claude.com/resources/tutorials',
 'https://claude.com/resources/use-cases',
 'https://www.anthropic.com/engineering',
 'https://docs.claude.com',
 'https://www.anthropic.com/company',
 'https://www.anthropic.com/careers',
 'https://www.anthropic.com/events',
 'https://www.anthropic.com/news',
 'https://claude.ai',
 'https://claude.com/product/overview',
 'https://claude.com/product/claude-code',
 'https://claude.com/product/cowork',
 'https://claude.com/platform/api',
 'https://claude.com/pricing',
 'https://claude.com/contact-sales',
 'https://www.anthropic.com/claude/opus',
 'https://www.anthropic.com/claude/sonnet',
 'https://www

## Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. 

In [8]:
link_system_prompt = """
    You are provided with a list of links found on a webpage.
    You are able to decide which of the links would be most relevant to include in a brochure about the company,
    such as links to an About page, or a Company page, or Careers/Jobs pages.
    You should respond in JSON as in this example:

    {
        "links": [
            {"type": "about page", "url": "https://full.url/goes/here/about"},
            {"type": "careers page", "url": "https://another.full.url/careers"}
        ]
    }
    """

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
        Here is the list of links on the website {url} -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):
    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    
    return user_prompt

In [9]:
print(get_links_user_prompt("https://anthropic.com"))


        Here is the list of links on the website https://anthropic.com -
        Please decide which of these are relevant web links for a brochure about the company, 
        respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, email links.

        Links (some might be relative links):
    #main
#footer
https://www.anthropic.com/
https://www.anthropic.com/research
https://www.anthropic.com/economic-futures
https://www.anthropic.com/constitution
https://www.anthropic.com/transparency
https://www.anthropic.com/responsible-scaling-policy
http://trust.anthropic.com/
https://www.anthropic.com/learn
https://claude.com/resources/tutorials
https://claude.com/resources/use-cases
https://www.anthropic.com/engineering
https://docs.claude.com
https://www.anthropic.com/company
https://www.anthropic.com/careers
https://www.anthropic.com/events
https://www.anthropic.com/news
https://claude.ai
https://claude.com/product/overview
https://claude.com/produc

In [20]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [21]:
select_relevant_links("https://anthropic.com")

Selecting relevant links for https://anthropic.com by calling llama3.2
Found 11 relevant links


{'links': [{'type': 'Company page', 'url': 'https://www.anthropic.com/'},
  {'type': 'About us page', 'url': 'https://docs.claude.com'},
  {'type': 'Transparency page',
   'url': 'https://www.anthropic.com/transparency'},
  {'type': 'Responsible scaling policy',
   'url': 'https://www.anthropic.com/responsible-scaling-policy'},
  {'type': 'Research page', 'url': 'https://www.anthropic.com/research'},
  {'type': 'Economic futures page',
   'url': 'https://www.anthropic.com/economic-futures'},
  {'type': 'Constitution page',
   'url': 'https://www.anthropic.com/constitution'},
  {'type': 'Careers página', 'url': 'https://www.anthropic.com/careers'},
  {'type': 'About us blog', 'url': 'https://www.claude.com/blog'},
  {'type': 'Community page', 'url': 'https://claude.com/community'},
  {'type': 'Learn section home page',
   'url': 'https://www.anthropic.com/learn'}]}

# Let's make a brochure now!

In [22]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [23]:
print(fetch_page_and_all_relevant_links("https://anthropic.com"))

Selecting relevant links for https://anthropic.com by calling llama3.2
Found 13 relevant links
## Landing Page:

Home \ Anthropic

Skip to main content
Skip to footer
Research
Economic Futures
Commitments
Initiatives
Claude's Constitution
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at Anthropic
Developer docs
Company
About
Careers
Events
News
Try Claude
Try Claude
Try Claude
Learn more about Claude
Products
Claude
Claude Code
Claude Cowork
Claude Platform
Pricing
Contact sales
Models
Opus
Sonnet
Haiku
Log in
Claude.ai
Claude Console
EN
This is some text inside of a div block.
Log in to Claude
Log in to Claude
Log in to Claude
Download app
Download app
Download app
Research
Economic Futures
Commitments
Initiatives
Claude's Constitution
Transparency
Responsible Scaling Policy
Trust center
Security and compliance
Learn
Learn
Anthropic Academy
Tutorials
Use cases
Engineering at Anthropic
Develope

In [ ]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model = MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream = True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [31]:
stream_brochure("Antropic", "https://anthropic.com")

Selecting relevant links for https://anthropic.com by calling llama3.2
Found 9 relevant links


**Anthropic: Empowering the Future of AI with Safety and Responsibility**

At Anthropic, we believe that Artificial Intelligence (AI) has the potential to revolutionize many aspects of our lives. However, it also poses significant risks and challenges that must be addressed. As a public benefit corporation, we are dedicated to ensuring that the benefits of AI are secured while mitigating its risks.

**Our Mission**

We conduct research and develop products that put safety at the forefront. Our goal is to create a future where AI enhances human life without compromising our values or the planet's well-being.

**Our Products**

Our flagship product, Claude Opus 4.6, is the world's most powerful model for coding, agents, and professional work. We also offer Claude Code, a platform that enables developers to build and deploy AI models with ease.

**Research Initiatives**

We invest in research initiatives that explore the impact of AI on society and develop innovative solutions to address its challenges. Our studies, such as the largest ever conducted on AI and its effects on lives around the world, provide insights into the benefits and risks of AI.

**Company Culture**

Our company values safety, transparency, and responsible scaling. We believe in creating a culture that fosters open discussions, respectful debates, and collaborative problem-solving. Our employees are passionate about making a positive impact with their work.

**Join Us**

We are seeking talented individuals who share our vision for a safe and responsible AI future. Join us as we shape the future of AI together.

### Careers

* Developers
* Researchers
* Engineers
* Policy Experts
* Community Builders

### Learn More About Claude

Download our app to experience the power of AI in your daily life.

Follow us on social media to stay up-to-date with our latest releases and initiatives.